# Train the "Hey Ollie" wake word model

This notebook trains a new openWakeWord model that detects **"Hey Ollie"**, to replace the bundled "Hey Jarvis" model in the Ollie extension.

It follows openWakeWord's automatic training pipeline (`openWakeWord/openwakeword/train.py`), in four steps:

1. **Generate** synthetic "hey ollie" speech with Piper TTS (many voices and speeds).
2. **Augment** those clips with room impulse responses and background noise.
3. **Train** a small DNN on openWakeWord's pre-computed speech features, with a large set of negative (non-wake-word) audio.
4. **Export** `hey_ollie.onnx`, then check it on test clips.

## Where to run it

- **Google Colab with a GPU (recommended).** Use a free T4 runtime. The pinned TensorFlow 2.8 needs Python 3.10, so check the runtime's Python version first; if it is newer, use a Python 3.10 environment on Linux.
- A **Linux machine with Python 3.10** and a GPU also works. `piper-phonemize` and the pinned TensorFlow build are Linux-only in practice.
- **Not macOS with Python 3.12**, which the pinned TensorFlow 2.8 does not support. Training on the Mac is slow and the pinned packages will fail to install.

## What it downloads

- Piper voice model, about 100 MB.
- MIT room impulse responses, a few hundred MB.
- An AudioSet slice, about 1 GB, for background noise.
- openWakeWord's pre-computed training features (about 2,000 hours of speech, ACAV100M) and the false-positive validation set. The training features file is large, so check you have the disk space first.

Expect several hours in total on a T4. Most of the time goes to steps 5 and 7. The upstream timings are a guide only; I have not timed this run.

Run the cells in order. If step 1 fails part way, run it again: it continues where it stopped.

## 1. Install the environment

Restart the runtime after this cell finishes, then run the rest of the notebook from the top.

In [ ]:
# Piper TTS, used to generate the synthetic "hey ollie" clips (Linux only)
!git clone -q https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize webrtcvad

# openWakeWord with its training dependencies
!git clone -q https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword

!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
!pip install -q audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install -q tensorflow-cpu==2.8.1 tensorflow_probability==0.16.0 onnx_tf==1.10.0
!pip install -q pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

# Models that openWakeWord needs at training time (Colab workaround from the upstream notebook)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

## 2. Download room impulse responses and background noise

Room impulse responses make the synthetic speech sound like it was said in different rooms. Background noise teaches the model to ignore everyday sounds.

In [ ]:
import os
from pathlib import Path
import numpy as np
import scipy
import datasets
from tqdm import tqdm

# MIT room impulse responses (https://mcdermottlab.mit.edu/Reverb/IR_Survey.html)
os.makedirs("./mit_rirs", exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset):
    name = row["audio"]["path"].split("/")[-1]
    scipy.io.wavfile.write(os.path.join("./mit_rirs", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

In [ ]:
# AudioSet, one part (https://research.google.com/audioset/dataset/index.html), converted to 16 kHz.
# For a stronger model, use more of AudioSet (https://huggingface.co/datasets/agkphysics/AudioSet)
# and add other noise sets (for example FSD50K or Freesound) to background_paths in the config.
os.makedirs("audioset", exist_ok=True)
fname = "bal_train09.tar"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -q -O audioset/{fname} {link}
!cd audioset && tar -xf bal_train09.tar

os.makedirs("./audioset_16k", exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(p) for p in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000, (row["audio"]["array"] * 32767).astype(np.int16))

## 3. Download pre-computed openWakeWord features

These features are openWakeWord's own embeddings of about 2,000 hours of speech and other audio (ACAV100M). The model trains against them as its negative examples. The validation set (about 11 hours of speech, noise and music) estimates how often the model fires by mistake.

In [ ]:
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 4. Configure the training run

The config is the same format as openWakeWord's `examples/custom_model.yml`, with the target changed to **"hey ollie"**.

Key settings:
- `target_phrase`: what the model should detect. Use the spelling and words people will actually say.
- `n_samples`: number of synthetic positive clips. 20,000 is the minimum upstream recommends; more usually helps.
- `steps`: maximum training steps. Training stops early when it stops improving.
- `target_false_positives_per_hour`: the trade-off between misses and false alarms. The extension's `WAKE_THRESHOLD` (0.5) is set later.

In [ ]:
import yaml

config = yaml.safe_load(open("openwakeword/examples/custom_model.yml"))

config["model_name"] = "hey_ollie"
config["target_phrase"] = ["hey ollie"]
config["custom_negative_phrases"] = []           # add words the model should ignore, e.g. "hey olly" if needed
config["n_samples"] = 20000
config["n_samples_val"] = 2000
config["steps"] = 50000
config["target_false_positives_per_hour"] = 0.2

config["output_dir"] = "./my_custom_model"
config["piper_sample_generator_path"] = "./piper-sample-generator"
config["rir_paths"] = ["./mit_rirs"]
config["background_paths"] = ["./audioset_16k"]
config["background_paths_duplication_rate"] = [1]
config["false_positive_validation_data_path"] = "./validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open("hey_ollie.yaml", "w") as f:
    yaml.dump(config, f)

print(open("hey_ollie.yaml").read()[:1200])

## 5. Generate the synthetic "hey ollie" clips

If this step fails part way, run the cell again. It continues until the target number of clips exists.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config hey_ollie.yaml --generate_clips

## 6. Augment the clips

Mixes each clip with a random room and background noise.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config hey_ollie.yaml --augment_clips

## 7. Train the model

This is the longest step. Training stops early once the model stops improving on the validation data.

In [ ]:
!python openwakeword/openwakeword/train.py --training_config hey_ollie.yaml --train_model

## 8. Export and copy into the extension

Training writes `my_custom_model/hey_ollie.onnx`. The extension loads this file from `models/wakeword/` in the repo.

This cell copies it into the repo folder and names it `hey_ollie_v0.1.onnx`, following the naming of the bundled model. Download the file from Colab first if you run this notebook outside the repo.

In [ ]:
import hashlib, shutil, os

src = "./my_custom_model/hey_ollie.onnx"
assert os.path.exists(src), "Training did not produce the model. Check the output of step 7."

digest = hashlib.sha256(open(src, "rb").read()).hexdigest()
print("sha256", digest)
print("size", os.path.getsize(src), "bytes")

# On Colab, download the file instead of copying it to the repo:
# from google.colab import files; files.download(src)

## 9. Check the model on test clips

Put short recordings of "Hey Ollie" in `test_clips/positive/` and recordings of other speech in `test_clips/negative/`. Use 16 kHz mono 16-bit WAV files, the same format as `test/fixtures/audio/` in the repo.

The model should fire on the positive clips and stay quiet on the negative ones. Compare the numbers with the threshold the extension uses (`WAKE_THRESHOLD` = 0.5).

In [ ]:
import glob
import wave
import numpy as np
from openwakeword.model import Model

model = Model(wakeword_models=["./my_custom_model/hey_ollie.onnx"], inference_framework="onnx")

def max_score(path):
    with wave.open(path, "rb") as w:
        assert w.getframerate() == 16000 and w.getnchannels() == 1 and w.getsampwidth() == 2, path
        audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    model.reset()
    best = 0.0
    for i in range(0, len(audio) - 1280 + 1, 1280):  # 80 ms frames
        scores = model.predict(audio[i:i + 1280])
        best = max(best, max(scores.values()))
    return best

for label in ["positive", "negative"]:
    files = sorted(glob.glob(f"./test_clips/{label}/*.wav"))
    for f in files:
        print(f"{label:8s} {max_score(f):.3f}  {os.path.basename(f)}")
    if not files:
        print(f"(no clips in test_clips/{label}/)")